# Inspecting & data Cleaning 🧹

## import Libs

In [1]:
# ==========================================
# 🎨 Cell 1: Environment Setup & Professional Theme
# ==========================================

# 📦 1. Install Required Packages
# ==========================================
!pip install -q pandas numpy matplotlib seaborn plotly scipy scikit-learn \
    ydata-profiling missingno sqlalchemy psycopg2-binary rapidfuzz \
    ipywidgets ipython-sql

# ==========================================
# 📚 2. Import Libraries
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import missingno as msno
from scipy import stats
import warnings
import sqlalchemy
from google.colab import userdata
from IPython.display import display, Markdown, HTML

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

# ==========================================
# 🎨 3. Professional Theme Configuration
# ==========================================
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Plotly Custom Theme
custom_theme = go.layout.Template(
    layout=go.Layout(
        font=dict(family='Inter, sans-serif', size=12, color='#1e293b'),
        paper_bgcolor='#ffffff',
        plot_bgcolor='#f8fafc',
        title=dict(font=dict(size=18, color='#0f172a'), x=0.5),
        xaxis=dict(
            gridcolor='#e2e8f0',
            zerolinecolor='#cbd5e1',
            title=dict(font=dict(size=14))
        ),
        yaxis=dict(
            gridcolor='#e2e8f0',
            zerolinecolor='#cbd5e1',
            title=dict(font=dict(size=14))
        ),
        legend=dict(
            orientation='h',
            yanchor='bottom',
            y=1.02,
            xanchor='right',
            x=1
        )
    )
)

# ✅ ✅ ✅ التعديل هنا: تسجيل القالب أولاً، ثم تعيينه كافتراضي ✅ ✅ ✅
pio.templates["custom_theme"] = custom_theme
pio.templates.default = "custom_theme"

# ==========================================
# ✅ 4. Confirmation
# ==========================================
display(Markdown("""
# 🏢 Bayut Real Estate Analytics
## 📊 Phase 1: Data Inspection & Cleaning

**Environment:** Ready ✅
**Libraries:** Loaded ✅
**Theme:** Professional ✅
**Status:** Ready to connect to Supabase 🚀
"""))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.5/682.5 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 2.8 MB/s eta 0:00:00



# 🏢 Bayut Real Estate Analytics
## 📊 Phase 1: Data Inspection & Cleaning

**Environment:** Ready ✅
**Libraries:** Loaded ✅
**Theme:** Professional ✅
**Status:** Ready to connect to Supabase 🚀


## importing data from DB

In [2]:
# ==========================================
# 🗄️ Cell 2: Secure Connection to Supabase
# ==========================================

# 🔐 1. Fetch Password from Colab Secrets
# ==========================================
try:
    DB_PASSWORD = userdata.get('SUPABASE_PASSWORD')
    if not DB_PASSWORD:
        raise ValueError("SUPABASE_PASSWORD not found in Colab Secrets.")
    display(Markdown("✅ **Authentication successful:** Password retrieved securely from Secrets."))
except Exception as e:
    display(Markdown(f"""
    ❌ **Authentication Error:** {e}

    💡 **Tip:**
    1. Click on 🔑 Secrets icon (left sidebar)
    2. Add a new secret named `SUPABASE_PASSWORD`
    3. Toggle **Notebook access** to ON
    """))
    DB_PASSWORD = ""

# ==========================================
# 🔌 2. Connection Configuration (Transaction Pooler)
# ==========================================
# ⚠️ هذه بياناتك الحقيقية من المحادثات السابقة
DB_USER = userdata.get("SUPABASE_USER")
DB_HOST = "aws-1-eu-west-1.pooler.supabase.com"
DB_PORT = "6543"  # Transaction Pooler Port
DB_NAME = "postgres"

# Build Connection URI
DATABASE_URI = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

display(Markdown(f"""
### 🔌 Connection Details:
- **Host:** `{DB_HOST}`
- **Port:** `{DB_PORT}` (Transaction Pooler)
- **Database:** `{DB_NAME}`
- **User:** `{'secret info'}`
"""))

# ==========================================
# 📥 3. Fetch Raw Data from Supabase
# ==========================================
try:
    display(Markdown("### 🔄 Connecting to Supabase & Fetching Raw Data..."))

    # Create SQLAlchemy Engine
    engine = sqlalchemy.create_engine(DATABASE_URI)

    # Fetch ALL raw data (for Time-Series analysis)
    query = "SELECT * FROM properties ORDER BY scraped_at DESC;"
    df = pd.read_sql(query, engine)

    # Free up the pooler connection immediately
    engine.dispose()

    display(Markdown("### 🎉 Data Loaded Successfully!"))
    display(Markdown(f"**📊 Dataset Dimensions:** `{df.shape[0]:,}` rows × `{df.shape[1]}` columns"))

except Exception as e:
    display(Markdown(f"❌ **Connection Error:** {e}"))
    df = pd.DataFrame()

✅ **Authentication successful:** Password retrieved securely from Secrets.


### 🔌 Connection Details:
- **Host:** `aws-1-eu-west-1.pooler.supabase.com`
- **Port:** `6543` (Transaction Pooler)
- **Database:** `postgres`
- **User:** `secret info`


### 🔄 Connecting to Supabase & Fetching Raw Data...

### 🎉 Data Loaded Successfully!

**📊 Dataset Dimensions:** `17,042` rows × `40` columns

## General

In [3]:
df.head()

,id,scraped_at,url,property_img,price,currency,title,location,beds,baths,area,property_type,purpose,reference_no,completion,furnishing,trucheck_date,added_on,handover_date,description,amenities,building_name,floors,retail_centres,swimming_pools,parking_spaces,building_area,elevators,agent_name,agency_name,developer,ownership,built_up_area,balcony_size,parking_availability,permit_number,zone_name,registered_agency,rera,brn
0,17043,2026-09-01 08:57:25.562363,https://www.bayut.com/property/details-1578507...,https://images.bayut.com/thumbnails/851575112-...,"2,719,000",AED,Luxury Residence | Prime Location | Investor Deal,"Hado by Beyond Tower C, Hado by Beyond, Dubai ...",1,2,"1,074 sqft",Apartment,For Sale,Bayut - 102164-VCQUKu,Off-Plan,Unfurnished,10 July 2026,10 July 2026,Q2 2029,House & Hedges Real Estate is pleased to offer...,"Parking Spaces, : 1, Centrally Air-Conditioned...",Hado By Beyond Tower C,23,7,1,867,"617,308 sqft",4,Syed Jalil Hussain,House and Hedges Real Estate,SANDY SHORES REAL ESTATE L.L.C S.O.C,Freehold,"1,074 sqft",343 sqft,Yes,None,Palm Deira,HOUSE AND HEDGES REAL ESTATE L.L.C,1391086,80558
1,17042,2026-09-01 08:57:25.083745,https://www.bayut.com/property/details-1577870...,https://images.bayut.com/thumbnails/851465284-...,"5,500,000",AED,Under Original Price | Exclusive | Sea View,"Th8, The Crescent, Palm Jumeirah, Dubai",2,2,"1,214 sqft",Apartment,For Sale,Bayut - A1-S-10348769,Ready,Furnished,None,10 July 2026,None,This luxury beachfront 2-bedroom apartment is ...,"Furnished, Parking Spaces, Balcony or Terrace,...",THE 8,9,1,3,516,"1,345,654 sqft",14,Michaela Meier,A1 Properties,C FOURTEEN FZE,Freehold,"1,214 sqft",106 sqft,Yes,None,Palm Jumeirah,A1 PROPERTIES L.L.C,12095,55402
2,17041,2026-09-01 08:57:23.853692,https://www.bayut.com/property/details-1574647...,https://images.bayut.com/thumbnails/855995469-...,"578,000",AED,STUIO FOR SELL DIRECT FROM DEVELOPER-NO COMMIS...,"RR Grand, Residential District, Dubai South, D...",Studio,1,343 sqft,Apartment,For Sale,Bayut - 109716-yZUlaI,Off-Plan,Furnished,None,7 July 2026,Q2 2027,"Studio Apartment for Sale | RR Grand, Dubai So...","Centrally Air-Conditioned, Central Heating, Do...",RR Grand,6,None,1,76,"118,067 sqft",3,Mohammad Danish Shareef,Blanco Thornton Properties,RED ROSE PROPERTIES L.L.C,None,None,None,None,None,Madinat Al Mataar,BLANCO THORNTON PROPERTIES,29046,28294
3,17040,2026-09-01 08:57:22.573414,https://www.bayut.com/property/details-1577232...,https://images.bayut.com/thumbnails/851338212-...,"2,100,000",AED,Burj Khalifa Views | Huge Terrace | 1BHK Apart...,"Injazzat Residence, Meydan Avenue, Meydan, Dubai",1,2,"1,144 sqft",Apartment,For Sale,Bayut - 109279-9hdDi6,Ready,Furnished,None,9 July 2026,None,Aylesford Middle East is delighted to offer t...,"Furnished, Parking Spaces, : 1, Balcony or Ter...",Injazzat Residence,6,3,1,63,"141,344 sqft",3,Atif Wasim Chishti,Aylesford Middle East One Properties,None,Freehold,"1,144 sqft",381 sqft,Yes,None,Nad Al Shiba First,AYLESFORD MIDDLE EAST ONE PROPERTIES L.L.C,53346,91173
4,17039,2026-09-01 08:56:56.779538,https://www.bayut.com/property/details-1570884...,https://images.bayut.com/thumbnails/850069179-...,"815,284",AED,SPACIOUS 1 BEDROOM APARTMENT | MODERN LIVING |...,"Kappa Acca 1, Dubai South, Dubai",1,2,747 sqft,Apartment,For Sale,Bayut - Nadeem-Acca1-1BHK,Ready,Furnished,None,4 July 2026,None,Homes Partner Real Estate is delighted to pres...,"Balcony or Terrace, Lobby in Building, Gym or ...",KAPPA ACCA 1,7,None,None,72,"96,895 sqft",None,Muhammad Nadeem,Homes Partner Real Estate,None,Freehold,747 sqft,160 sqft,Yes,None,Madinat Al Mataar,HOMES PARTNER REAL ESTATE MANAGEMENT SUPERVISI...,23374,79456


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17042 entries, 0 to 17041
Data columns (total 40 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   id                    17042 non-null  int64         
 1   scraped_at            17042 non-null  datetime64[ns]
 2   url                   17042 non-null  object        
 3   property_img          17042 non-null  object        
 4   price                 17042 non-null  object        
 5   currency              17042 non-null  object        
 6   title                 17042 non-null  object        
 7   location              17042 non-null  object        
 8   beds                  16226 non-null  object        
 9   baths                 16227 non-null  object        
 10  area                  17042 non-null  object        
 11  property_type         17042 non-null  object        
 12  purpose               17042 non-null  object        
 13  reference_no    

In [5]:
df.shape

(17042, 40)

- we have missing values

In [6]:
df.isnull().sum()

,0
id,0
scraped_at,0
url,0
property_img,0
price,0
currency,0
title,0
location,0
beds,816
baths,815


## Comprehensive Inspector 🕵🏻
<mark>Here will check each column.</mark>

In [7]:
#==============================
#======= Data Inspector =======
#==============================

def comprehensive_data_inspector(df, max_samples=5):
    """
    دالة احترافية لفحص شامل لجميع أعمدة الـ DataFrame.
    تعرض: نوع البيانات، عدد القيم الفارغة، النسبة المئوية، عدد القيم الفريدة، وعينة من القيم.
    """
    display(Markdown("## 🔍 Comprehensive Data Inspector (فحص شامل للبيانات)"))
    display(Markdown(f"**إجمالي الصفوف:** `{len(df):,}` | **إجمالي الأعمدة:** `{len(df.columns)}`"))

    summary_data = []
    total_rows = len(df)

    for col in df.columns:
        # 1. حساب القيم الفارغة (يشمل NaN و Empty Strings)
        missing_count = df[col].isna().sum() + (df[col].astype(str).str.strip() == '').sum()
        missing_pct = (missing_count / total_rows) * 100 if total_rows > 0 else 0

        # 2. حساب القيم الفريدة (بدون الـ NaN)
        unique_count = df[col].nunique(dropna=True)

        # 3. استخراج عينة من القيم الفريدة للعرض
        unique_vals = df[col].dropna().astype(str).unique()
        # تنظيف العينات من السلاسل الفارغة تماماً
        unique_vals = [v for v in unique_vals if v.strip() != '' and v.lower() != 'nan']

        samples = unique_vals[:max_samples]
        samples_str = ", ".join([f"`{v}`" for v in samples])

        if len(unique_vals) > max_samples:
            samples_str += f" <span style='color:gray'>(+{len(unique_vals) - max_samples} more)</span>"
        elif len(unique_vals) == 0:
            samples_str = "<span style='color:red'>All Empty/NaN</span>"

        summary_data.append({
            'Column': col,
            'Dtype': str(df[col].dtype),
            'Total': total_rows,
            'Missing': missing_count,
            'Missing %': missing_pct,
            'Unique': unique_count,
            'Sample Values': samples_str
        })

    # تحويل البيانات إلى DataFrame
    summary_df = pd.DataFrame(summary_data)

    # ترتيب الأعمدة لجعلها أسهل في القراءة
    summary_df = summary_df[['Column', 'Dtype', 'Total', 'Missing', 'Missing %', 'Unique', 'Sample Values']]

    # ترتيب الجدول تنازلياً حسب نسبة القيم المفقودة (الأكثر مشكلة أولاً)
    summary_df = summary_df.sort_values(by='Missing %', ascending=False).reset_index(drop=True)

    # ==========================================
    # 🎨 تنسيق الجدول ليبدو احترافياً (Styling)
    # ==========================================
    def highlight_missing(val):
        if val >= 50: return 'background-color: #fee2e2; color: #991b1b; font-weight: bold' # أحمر غامق (>50%)
        elif val >= 20: return 'background-color: #fef3c7; color: #92400e' # أصفر (20-50%)
        elif val > 0: return 'background-color: #d1fae5; color: #065f46' # أخضر فاتح (>0%)
        return ''

    styled_df = (
        summary_df.style
        .format({'Missing %': '{:.2f}%'})
        .background_gradient(cmap='Reds', subset=['Missing %'], vmin=0, vmax=100)
        .applymap(highlight_missing, subset=['Missing %'])
        .set_properties(**{'text-align': 'left', 'font-family': 'monospace', 'font-size': '13px'})
        .set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#f1f5f9'), ('color', '#0f172a'), ('font-weight', 'bold'), ('text-align', 'center')]},
            {'selector': 'td', 'props': [('border-bottom', '1px solid #e2e8f0')]}
        ])
        .set_caption("📊 Data Health & Uniqueness Report")
    )

    display(styled_df)

    # ==========================================
    # 💡 ملاحظات ذكية تلقائية (Smart Insights)
    # ==========================================
    display(Markdown("### 💡 ملاحظات ذكية (Smart Insights):"))

    # 1. أعمدة فارغة تماماً أو شبه فارغة
    high_missing = summary_df[summary_df['Missing %'] > 80]['Column'].tolist()
    if high_missing:
        display(Markdown(f"⚠️ **أعمدة شبه فارغة (>80%):** `{', '.join(high_missing)}` → *يُفضل حذفها أو دمجها مع أعمدة أخرى.*"))

    # 2. أعمدة ذات قيم فريدة قليلة جداً (مناسبة لـ One-Hot Encoding)
    low_unique = summary_df[(summary_df['Unique'] <= 5) & (summary_df['Unique'] > 1) & (summary_df['Missing %'] < 20)]['Column'].tolist()
    if low_unique:
        display(Markdown(f"✅ **أعمدة فئوية ممتازة (Low Cardinality):** `{', '.join(low_unique)}` → *مثالية لـ One-Hot Encoding أو Target Encoding.*"))

    # 3. أعمدة ذات قيم فريدة عالية جداً (High Cardinality)
    high_unique = summary_df[summary_df['Unique'] > 100]['Column'].tolist()
    if high_unique:
        display(Markdown(f"🔍 **أعمدة ذات تنوع عالي (High Cardinality):** `{', '.join(high_unique[:5])}...` → *تحتاج إلى تنظيف نصوص أو Target Encoding.*"))

# ==========================================
# 🚀 تشغيل الدالة على بياناتك
# ==========================================
# تأكد أن الـ DataFrame الخاص بك اسمه 'df'
comprehensive_data_inspector(df, max_samples=6)

## 🔍 Comprehensive Data Inspector (فحص شامل للبيانات)

**إجمالي الصفوف:** `17,042` | **إجمالي الأعمدة:** `40`

,Column,Dtype,Total,Missing,Missing %,Unique,Sample Values
0,permit_number,object,17042,15388,90.29%,907,"`202641000240392`, `20260000960493`, `20260000953665-2`, `20260000957734`, `20260000966598`, `20260000953225` (+901 more)"
1,retail_centres,object,17042,14135,82.94%,39,"`7`, `1`, `3`, `5`, `6`, `2` (+33 more)"
2,swimming_pools,object,17042,12665,74.32%,9,"`1`, `3`, `2`, `4`, `8`, `5` (+3 more)"
3,elevators,object,17042,12204,71.61%,32,"`4`, `14`, `3`, `5`, `10`, `7` (+26 more)"
4,balcony_size,object,17042,12121,71.12%,639,"`343 sqft`, `106 sqft`, `381 sqft`, `160 sqft`, `688 sqft`, `124 sqft` (+633 more)"
5,parking_spaces,object,17042,12051,70.71%,707,"`867`, `516`, `76`, `63`, `72`, `462` (+701 more)"
6,parking_availability,object,17042,12000,70.41%,1,`Yes`
7,building_area,object,17042,11517,67.58%,1975,"`617,308 sqft`, `1,345,654 sqft`, `118,067 sqft`, `141,344 sqft`, `96,895 sqft`, `580,006 sqft` (+1969 more)"
8,floors,object,17042,11454,67.21%,96,"`23`, `9`, `6`, `7`, `16`, `20` (+90 more)"
9,building_name,object,17042,11436,67.10%,2025,"`Hado By Beyond Tower C`, `THE 8`, `RR Grand`, `Injazzat Residence`, `KAPPA ACCA 1`, `AZIZI PLAZA` (+2019 more)"


### 💡 ملاحظات ذكية (Smart Insights):

⚠️ **أعمدة شبه فارغة (>80%):** `permit_number, retail_centres` → *يُفضل حذفها أو دمجها مع أعمدة أخرى.*

✅ **أعمدة فئوية ممتازة (Low Cardinality):** `furnishing, completion` → *مثالية لـ One-Hot Encoding أو Target Encoding.*

🔍 **أعمدة ذات تنوع عالي (High Cardinality):** `permit_number, balcony_size, parking_spaces, building_area, building_name...` → *تحتاج إلى تنظيف نصوص أو Target Encoding.*

## Investigating missing values 🔍

### baths 🛁 & beds 🛌

In [8]:
null_baths = df.loc[df['baths'].isnull()]
null_beds = df.loc[df['beds'].isnull()]


In [9]:
null_baths

,id,scraped_at,url,property_img,price,currency,title,location,beds,baths,area,property_type,purpose,reference_no,completion,furnishing,trucheck_date,added_on,handover_date,description,amenities,building_name,floors,retail_centres,swimming_pools,parking_spaces,building_area,elevators,agent_name,agency_name,developer,ownership,built_up_area,balcony_size,parking_availability,permit_number,zone_name,registered_agency,rera,brn
28,17015,2026-09-01 08:53:37.313229,https://www.bayut.com/property/details-1571902...,https://images.bayut.com/thumbnails/849830558-...,"880,000",AED,Residential land for sale (G+2)prime location ...,"Al Zubair Orchards, Al Zubair, Sharjah",None,None,"6,110 sqft",Residential Plot,For Sale,Bayut - 9123-XtEVYD,Ready,None,None,5 July 2026,None,"Residential land for sale (G+2), prime locatio...",Freehold,None,None,None,None,None,None,None,Ali Mohamed Ali,Al Mutamaiz Real Estate,None,None,None,None,None,None,None,None,None,None
34,17009,2026-09-01 08:53:09.276542,https://www.bayut.com/property/details-1584149...,https://images.bayut.com/thumbnails/852603235-...,"1,350,000",AED,SPECIOUS corner PLOT AL HELIO 1 AVAILBLE FOR SALE,"Al Helio 1, Al Helio, Ajman",None,None,"5,400 sqft",Residential Plot,For Sale,Bayut - 105099-4JawGf,Ready,None,None,15 July 2026,None,"WELCOME TO QSA PROPERTIES! Al Helio 2, Ajman. ...","Centrally Air-Conditioned, Central Heating, Do...",None,None,None,None,None,None,None,Sheraz Sultan,QSA Properties,None,None,None,None,None,None,124674,None,None,None
52,16991,2026-09-01 08:51:21.372861,https://www.bayut.com/property/details-1467285...,https://images.bayut.com/thumbnails/828378245-...,"185,000",AED,One of the most beautiful and peaceful areas i...,"Masfout, Ajman",None,None,"4,400 sqft",Residential Plot,For Sale,Bayut - 9123-tBzzTi,Off-Plan,None,None,29 March 2026,None,A great opportunity to own a residential plot ...,"Waste Disposal, Maintenance Staff, Cleaning Se...",None,None,None,None,None,None,None,Eman Ibrahim,Al Mutamaiz Real Estate,None,None,None,None,None,None,None,None,None,None
53,16990,2026-09-01 08:51:20.862969,https://www.bayut.com/property/details-1465397...,https://images.bayut.com/thumbnails/828064490-...,"2,100,000",AED,Prime G+3 Residential Plot in Tilal City – (Ne...,"Tilal City, Sharjah",None,None,"12,165 sqft",Residential Plot,For Sale,Bayut - 6348-Yx9AAg,Ready,None,None,26 March 2026,None,An excellent investment opportunity in the hig...,None,None,None,None,None,None,None,None,Khizer Mahenti,Mahenti Real Estate,None,None,None,None,None,None,1896,None,None,32161
69,16974,2026-09-01 08:48:54.008112,https://www.bayut.com/property/details-1474062...,https://images.bayut.com/thumbnails/829556088-...,"140,000",AED,Plot for Sale – Manama 14 (Ajman),"Al Manama, Ajman",None,None,"1,721 sqft",Residential Plot,For Sale,Bayut - 102733-SJ5o7N,Ready,None,None,4 April 2026,None,Plot for Sale – Manama 14 (Ajman) أرض للبيع –...,"Lobby in Building, Kids Play Area, Lawn or Gar...",None,None,None,None,None,None,None,Noman khan Firdous Khan,Al Sheikh Brothers Real Estate,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16470,572,2026-08-30 17:13:01.401567,https://www.bayut.com/property/details-1620338...,https://images.bayut.com/thumbnails/860346350-...,"1,999,999",AED,Huge Plot | Single Row | Build Your Dream Home!!!,"Al Reeman 2, Al Shamkha, Abu Dhabi",None,None,"5,490 sqft",Residential Plot,For Sale,Bayut - 10678-PV5IXG,Off-Plan,None,27 August 2026,17 August 2026,None,AlReeman masterfully integrates contemporary d...,"Gym or Health Club, Swimming Pool, Day Care Ce...",None,None,None,None,None,None,None,Mahmoud Mohsen,Oia Properties,None,None,None,None,None,20260001014810,CN-3990375,"202402174045, 202402732607",None,None
16474,568,2026-08-30 17:12:48.232112,https://www.bayut.com/property/details-1582943...,https://images.bayu

In [10]:
null_baths['property_type'].unique()

array(['Residential Plot', 'Residential Building'], dtype=object)

In [11]:
null_baths['url'][:5].values

array(['https://www.bayut.com/property/details-15719022.html',
       'https://www.bayut.com/property/details-15841498.html',
       'https://www.bayut.com/property/details-14672853.html',
       'https://www.bayut.com/property/details-14653970.html',
       'https://www.bayut.com/property/details-14740622.html'],
      dtype=object)

- not found any info.

**Empty in case:**
- 'Residential Plot', 'Residential Building'

**Check if there are some property belong to 'Residential Plot', 'Residential Building', but not null**

In [12]:
df.loc[(df['baths'].notnull()) & (df['property_type'].isin(['Residential Plot', 'Residential Building']))]

,id,scraped_at,url,property_img,price,currency,title,location,beds,baths,area,property_type,purpose,reference_no,completion,furnishing,trucheck_date,added_on,handover_date,description,amenities,building_name,floors,retail_centres,swimming_pools,parking_spaces,building_area,elevators,agent_name,agency_name,developer,ownership,built_up_area,balcony_size,parking_availability,permit_number,zone_name,registered_agency,rera,brn
1007,16036,2026-09-01 06:52:32.596622,https://www.bayut.com/property/details-1629055...,https://images.bayut.com/thumbnails/861993694-...,"10,000,000",AED,Prime Commercial Building for Sale on Main Road,"Al Helio 2, Al Helio, Ajman",3,4,"9,030 sqft",Residential Building,For Sale,Bayut - awtan-S-09437,Ready,None,None,25 August 2026,None,"Commercial Building for Sale in Al Helio 2, Aj...",None,None,None,None,None,None,None,None,Amal Arraki,Awtan Real Estate,None,None,None,None,None,None,None,None,None,None
1115,15928,2026-09-01 06:39:29.354813,https://www.bayut.com/property/details-1622145...,https://images.bayut.com/thumbnails/860706606-...,"29,000,000",AED,"Residential Commercial Building – Al Rawda, Ajman","Al Rawda 2, Al Rawda, Ajman",10+,10+,"10,200 sqft",Residential Building,For Sale,Bayut - AWTAN-S-04954,Ready,None,None,19 August 2026,None,# Residential & Commercial Building for Sale –...,Lobby in Building,None,None,None,None,None,None,None,Mohamed Mansour,Awtan Real Estate,None,None,None,None,None,None,None,None,None,None
1446,15597,2026-09-01 05:57:29.267256,https://www.bayut.com/property/details-1630869...,https://images.bayut.com/thumbnails/862353246-...,"7,200,000",AED,Corner Commercial Residential Building for Sal...,"Al Hamidiya 1, Al Hamidiyah, Ajman",10+,10+,"6,727 sqft",Residential Building,For Sale,Bayut - 104169-hossu,Ready,None,None,27 August 2026,None,An exceptional investment opportunity awaits s...,"Parking Spaces, : 1, Centrally Air-Conditioned...",None,None,None,None,None,None,None,Hossam Hassan,Queen Properties,None,None,None,None,None,None,887875,None,None,None
1448,15595,2026-09-01 05:57:16.043344,https://www.bayut.com/property/details-1630912...,https://images.bayut.com/thumbnails/862359979-...,"4,800,000",AED,High ROI Investment: Brand New Fully Furnished...,"Al Alia, Ajman",10+,10+,"8,085 sqft",Residential Building,For Sale,Bayut - 104169-hossA,Ready,Furnished,None,27 August 2026,None,An exceptional investment opportunity in the h...,"Furnished, Parking Spaces, : 1, Centrally Air-...",None,None,None,None,None,None,None,Hossam Hassan,Queen Properties,None,None,None,None,None,None,887875,None,None,None
1599,15444,2026-09-01 05:39:40.998236,https://www.bayut.com/property/details-1619172...,https://images.bayut.com/thumbnails/860118689-...,"4,000,000",AED,"A shop for sale in the Emirate of Sharjah, Al ...","Bu Tina, Sharjah",10+,10+,"2,200 sqft",Residential Building,For Sale,Bayut - 6788-686ft5,Ready,None,None,16 August 2026,None,Building for sale in Al-Boutaina Great locatio...,None,None,None,None,None,None,None,None,Bayan abood,Rukn Al Fakhama Real Estate,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12299,4743,2026-08-31 05:06:39.758531,https://www.bayut.com/property/details-1631517...,https://images.bayut.com/thumbnails/862469201-...,"1,700,000",AED,"Unleased, an empty building / strategic locati...","Al Nuaimiya, Ajman",6,9,"3,600 sqft",Residential Building,For Sale,Bayut - 107117-1gZQcm,Ready,None,None,27 August 2026,None,FOR SALE – Residential Building in Al Nuaimiya...,None,None,None,None,None,None,None,None,Jana Mohamed,Nasim Real Estate,None,None,None,None,None,None,131443,None,None,None
13740,3302,2026-08-31 02:08:56.669704,https://www.bayut.com/property/details-1633177...,https://images.bayut.com/thumbnails/862843214-...,"17,000,000",AED,Residential & Commercial Building for Sale | 1...,"Al Yasmeen, Ajman",7

In [13]:
# check some url
df.loc[(df['baths'].notnull()) & (df['property_type'].isin(['Residential Plot', 'Residential Building']))]['url'][:5].values

array(['https://www.bayut.com/property/details-16290559.html',
       'https://www.bayut.com/property/details-16221453.html',
       'https://www.bayut.com/property/details-16308693.html',
       'https://www.bayut.com/property/details-16309125.html',
       'https://www.bayut.com/property/details-16191728.html'],
      dtype=object)

- beds

In [14]:
null_beds['property_type'].unique()

array(['Residential Plot', 'Residential Building'], dtype=object)

In [15]:
null_beds['url'][:5].values

array(['https://www.bayut.com/property/details-15719022.html',
       'https://www.bayut.com/property/details-15841498.html',
       'https://www.bayut.com/property/details-14672853.html',
       'https://www.bayut.com/property/details-14653970.html',
       'https://www.bayut.com/property/details-14740622.html'],
      dtype=object)

- not found any info.

In [16]:
df.loc[(df['beds'].notnull()) & (df['property_type'].isin(['Residential Plot', 'Residential Building']))]

,id,scraped_at,url,property_img,price,currency,title,location,beds,baths,area,property_type,purpose,reference_no,completion,furnishing,trucheck_date,added_on,handover_date,description,amenities,building_name,floors,retail_centres,swimming_pools,parking_spaces,building_area,elevators,agent_name,agency_name,developer,ownership,built_up_area,balcony_size,parking_availability,permit_number,zone_name,registered_agency,rera,brn
1007,16036,2026-09-01 06:52:32.596622,https://www.bayut.com/property/details-1629055...,https://images.bayut.com/thumbnails/861993694-...,"10,000,000",AED,Prime Commercial Building for Sale on Main Road,"Al Helio 2, Al Helio, Ajman",3,4,"9,030 sqft",Residential Building,For Sale,Bayut - awtan-S-09437,Ready,None,None,25 August 2026,None,"Commercial Building for Sale in Al Helio 2, Aj...",None,None,None,None,None,None,None,None,Amal Arraki,Awtan Real Estate,None,None,None,None,None,None,None,None,None,None
1115,15928,2026-09-01 06:39:29.354813,https://www.bayut.com/property/details-1622145...,https://images.bayut.com/thumbnails/860706606-...,"29,000,000",AED,"Residential Commercial Building – Al Rawda, Ajman","Al Rawda 2, Al Rawda, Ajman",10+,10+,"10,200 sqft",Residential Building,For Sale,Bayut - AWTAN-S-04954,Ready,None,None,19 August 2026,None,# Residential & Commercial Building for Sale –...,Lobby in Building,None,None,None,None,None,None,None,Mohamed Mansour,Awtan Real Estate,None,None,None,None,None,None,None,None,None,None
1446,15597,2026-09-01 05:57:29.267256,https://www.bayut.com/property/details-1630869...,https://images.bayut.com/thumbnails/862353246-...,"7,200,000",AED,Corner Commercial Residential Building for Sal...,"Al Hamidiya 1, Al Hamidiyah, Ajman",10+,10+,"6,727 sqft",Residential Building,For Sale,Bayut - 104169-hossu,Ready,None,None,27 August 2026,None,An exceptional investment opportunity awaits s...,"Parking Spaces, : 1, Centrally Air-Conditioned...",None,None,None,None,None,None,None,Hossam Hassan,Queen Properties,None,None,None,None,None,None,887875,None,None,None
1448,15595,2026-09-01 05:57:16.043344,https://www.bayut.com/property/details-1630912...,https://images.bayut.com/thumbnails/862359979-...,"4,800,000",AED,High ROI Investment: Brand New Fully Furnished...,"Al Alia, Ajman",10+,10+,"8,085 sqft",Residential Building,For Sale,Bayut - 104169-hossA,Ready,Furnished,None,27 August 2026,None,An exceptional investment opportunity in the h...,"Furnished, Parking Spaces, : 1, Centrally Air-...",None,None,None,None,None,None,None,Hossam Hassan,Queen Properties,None,None,None,None,None,None,887875,None,None,None
1599,15444,2026-09-01 05:39:40.998236,https://www.bayut.com/property/details-1619172...,https://images.bayut.com/thumbnails/860118689-...,"4,000,000",AED,"A shop for sale in the Emirate of Sharjah, Al ...","Bu Tina, Sharjah",10+,10+,"2,200 sqft",Residential Building,For Sale,Bayut - 6788-686ft5,Ready,None,None,16 August 2026,None,Building for sale in Al-Boutaina Great locatio...,None,None,None,None,None,None,None,None,Bayan abood,Rukn Al Fakhama Real Estate,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12299,4743,2026-08-31 05:06:39.758531,https://www.bayut.com/property/details-1631517...,https://images.bayut.com/thumbnails/862469201-...,"1,700,000",AED,"Unleased, an empty building / strategic locati...","Al Nuaimiya, Ajman",6,9,"3,600 sqft",Residential Building,For Sale,Bayut - 107117-1gZQcm,Ready,None,None,27 August 2026,None,FOR SALE – Residential Building in Al Nuaimiya...,None,None,None,None,None,None,None,None,Jana Mohamed,Nasim Real Estate,None,None,None,None,None,None,131443,None,None,None
13740,3302,2026-08-31 02:08:56.669704,https://www.bayut.com/property/details-1633177...,https://images.bayut.com/thumbnails/862843214-...,"17,000,000",AED,Residential & Commercial Building for Sale | 1...,"Al Yasmeen, Ajman",7

**Note:**
I noticed that the missing values in the beds & baths are related to the property type in two cases: 'Residential Plot' and 'Residential Building'.
And here comes the role of business logic, indicating that this is a normal occurrence.

### amenities

- check from source

In [17]:
null_amenities=df.loc[df['amenities'].isnull()].reset_index()

In [18]:
null_amenities['url'][:5].values

array(['https://www.bayut.com/property/details-15750195.html',
       'https://www.bayut.com/property/details-15642879.html',
       'https://www.bayut.com/property/details-15701271.html',
       'https://www.bayut.com/property/details-15775352.html',
       'https://www.bayut.com/property/details-15848270.html'],
      dtype=object)

- after check the source, the info not Found.

- check description if found any info

In [19]:
import re
from IPython.display import display, Markdown

# ==========================================
# 🎯 1. قاموس المرافق الشامل (Amenity Dictionary)
# ==========================================
# كل مفتاح = اسم المرفق القياسي
# كل قيمة = قائمة بالمرادفات والصيغ المختلفة التي قد تظهر في الـ description
AMENITY_KEYWORDS = {
    'Swimming Pool': [
        r'\bswimming pool\b', r'\bpool\b', r'\bshared pool\b',
        r'\bprivate pool\b', r'\bcommunal pool\b', r'\binfinity pool\b',
        r'\brooftop pool\b', r'\bkids pool\b'
    ],
    'Gym': [
        r'\bgym\b', r'\bfitness\b', r'\bhealth club\b',
        r'\bfitness center\b', r'\bworkout\b', r'\bweight room\b'
    ],
    'Parking': [
        r'\bparking\b', r'\bcovered parking\b', r'\bparking space',
        r'\bgarage\b', r'\bvalet parking\b', r'\bbasement parking\b'
    ],
    'Security': [
        r'\bsecurity\b', r'\bcctv\b', r'\b24\/7 security\b',
        r'\bsecurity staff\b', r'\bsurveillance\b'
    ],
    'Concierge': [
        r'\bconcierge\b', r'\breception\b', r'\blobby\b',
        r'\bfront desk\b'
    ],
    'Kids Area': [
        r'\bkids play\b', r'\bplay area\b', r'\bchildren',
        r'\bday care\b', r'\bnursery\b', r'\bkids club\b'
    ],
    'Pets Allowed': [
        r'\bpets allowed\b', r'\bpet friendly\b', r'\ballows pets\b'
    ],
    'Maids Room': [
        r'\bmaids\b', r'\bmaid', r'\bservants\b', r'\bservant',
        r'\bstudy room\b', r'\bstudy\b'
    ],
    'Balcony': [
        r'\bbalcony\b', r'\bterrace\b', r'\bprivate terrace\b',
        r'\bbalcon', r'\bpatio\b'
    ],
    'Furnished': [
        r'\bfurnished\b', r'\bfully furnished\b', r'\bsemi[- ]furnished\b',
        r'\bwith furniture\b'
    ],
    'Central AC': [
        r'\bcentrally air[- ]conditioned\b', r'\bcentral ac\b',
        r'\bcentral air\b', r'\bhvac\b'
    ],
    'Sauna': [r'\bsauna\b'],
    'Steam Room': [r'\bsteam room\b', r'\bsteam\b'],
    'Jacuzzi': [r'\bjacuzzi\b', r'\bhot tub\b'],
    'Garden': [
        r'\bgarden\b', r'\blawn\b', r'\bgreen area\b', r'\blandscape\b'
    ],
    'BBQ Area': [r'\bbarbeque\b', r'\bbbq\b', r'\bgrill area\b'],
    'Business Center': [r'\bbusiness center\b', r'\bmeeting room\b'],
    'Laundry': [r'\blaundry\b', r'\bwashing\b'],
    'Broadband': [r'\bbroadband\b', r'\binternet\b', r'\bwifi\b'],
    'Elevator': [r'\belevator', r'\blift', r'\bservice elevator'],
    'Prayer Room': [r'\bprayer room\b', r'\bmosque\b'],
    'Private Pool': [r'\bprivate pool\b', r'\bown pool\b'],
    'Shared Pool': [r'\bshared pool\b', r'\bcommunal pool\b'],
}

# ==========================================
# 🛠️ 2. دالة الاستخراج الذكية
# ==========================================
def extract_amenities_from_description(df, text_column='description'):
    """
    تبحث في عمود description عن المرافق المخفية
    وترجع DataFrame مع أعمدة ثنائية (0/1) لكل مرفق
    """
    if text_column not in df.columns:
        print(f"❌ العمود '{text_column}' غير موجود!")
        return df

    print(f"🔍 بدء البحث في عمود '{text_column}' عن المرافق...")
    print(f"📊 إجمالي الصفوف: {len(df):,}")
    print(f"📝 الصفوف التي تحتوي على نص: {df[text_column].notna().sum():,}")

    # دمج النص وتحويله لحروف صغيرة
    text_corpus = df[text_column].fillna('').astype(str).str.lower()

    results = {}

    for amenity, patterns in AMENITY_KEYWORDS.items():
        # دمج كل الأنماط في regex واحد (OR)
        combined_pattern = '|'.join(patterns)

        # البحث باستخدام str.contains (vectorized = سريع جداً)
        matches = text_corpus.str.contains(combined_pattern, regex=True, na=False)

        # حفظ النتائج
        col_name = f"has_{amenity.lower().replace(' ', '_').replace('/', '_')}"
        results[col_name] = matches.astype(int)

        # طباعة الإحصائيات
        count = matches.sum()
        pct = (count / len(df)) * 100
        print(f"  ✅ {amenity:<20} : {count:>5,} ({pct:>5.2f}%)")

    # تحويل النتائج إلى DataFrame وإضافتها للـ df الأصلي
    amenities_df = pd.DataFrame(results, index=df.index)

    # دمج مع الـ DataFrame الأصلي
    df_enriched = pd.concat([df, amenities_df], axis=1)

    # حساب إجمالي المرافق لكل عقار
    amenity_cols = [c for c in amenities_df.columns if c.startswith('has_')]
    df_enriched['amenities_count_from_desc'] = df_enriched[amenity_cols].sum(axis=1)

    print(f"\n✨ تم استخراج {len(amenity_cols)} ميزة جديدة!")
    print(f"📊 متوسط المرافق لكل عقار: {df_enriched['amenities_count_from_desc'].mean():.2f}")

    return df_enriched

# ==========================================
# 🚀 3. تشغيل الدالة
# ==========================================
df = extract_amenities_from_description(df, 'description')

# ==========================================
# 📊 4. عرض النتائج
# ==========================================
display(Markdown("## 📊 ملخص المرافق المستخرجة من الـ Description"))

# جدول الإحصائيات
amenity_cols = [c for c in df.columns if c.startswith('has_')]
stats = pd.DataFrame({
    'Amenity': [c.replace('has_', '').replace('_', ' ').title() for c in amenity_cols],
    'Found': [df[c].sum() for c in amenity_cols],
    'Percentage': [f"{(df[c].mean() * 100):.2f}%" for c in amenity_cols]
}).sort_values('Found', ascending=False).reset_index(drop=True)

display(stats)

# ==========================================
# 🎨 5. رسم بياني
# ==========================================
import plotly.express as px

fig = px.bar(
    stats.head(15),
    x='Found',
    y='Amenity',
    orientation='h',
    title='🏊 Top 15 Amenities Extracted from Description',
    labels={'Found': 'Number of Properties', 'Amenity': 'Amenity'},
    color='Found',
    color_continuous_scale='Blues'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, width=900, height=600)
fig.show()

# ==========================================
# 🔎 6. أمثلة حية (عقارات وجد فيها المرافق)
# ==========================================
display(Markdown("## 🔎 أمثلة حية: عقارات تم اكتشاف مرافقها من الـ Description"))

# اختر مرفق واحد واعرض 3 أمثلة
sample_amenity = 'has_swimming_pool'
sample_df = df[df[sample_amenity] == 1][['title', 'price', sample_amenity, 'description']].head(3)

for idx, row in sample_df.iterrows():
    display(Markdown(f"""
### 🏠 {row['title']}
- **السعر:** `{row['price']}`
- **✅ يحتوي على:** Swimming Pool
- **📝 مقتطف من الوصف:**
> *{str(row['description'])[:300]}...*
"""))

🔍 بدء البحث في عمود 'description' عن المرافق...
📊 إجمالي الصفوف: 17,042
📝 الصفوف التي تحتوي على نص: 17,042
  ✅ Swimming Pool        : 8,537 (50.09%)
  ✅ Gym                  : 7,888 (46.29%)
  ✅ Parking              : 8,902 (52.24%)
  ✅ Security             : 6,052 (35.51%)
  ✅ Concierge            : 3,512 (20.61%)
  ✅ Kids Area            : 7,091 (41.61%)
  ✅ Pets Allowed         :   307 ( 1.80%)
  ✅ Maids Room           : 4,124 (24.20%)
  ✅ Balcony              : 7,057 (41.41%)
  ✅ Furnished            : 2,666 (15.64%)
  ✅ Central AC           : 1,140 ( 6.69%)
  ✅ Sauna                : 1,262 ( 7.41%)
  ✅ Steam Room           :   893 ( 5.24%)
  ✅ Jacuzzi              :   759 ( 4.45%)
  ✅ Garden               : 4,252 (24.95%)
  ✅ BBQ Area             : 2,371 (13.91%)
  ✅ Business Center      :   198 ( 1.16%)
  ✅ Laundry              : 1,253 ( 7.35%)
  ✅ Broadband            :   339 ( 1.99%)
  ✅ Elevator             : 1,794 (10.53%)
  ✅ Prayer Room          : 1,295 ( 7.60%)
  ✅ Private

## 📊 ملخص المرافق المستخرجة من الـ Description

,Amenity,Found,Percentage
0,Parking,8902,52.24%
1,Swimming Pool,8537,50.09%
2,Gym,7888,46.29%
3,Kids Area,7091,41.61%
4,Balcony,7057,41.41%
5,Security,6052,35.51%
6,Garden,4252,24.95%
7,Maids Room,4124,24.20%
8,Concierge,3512,20.61%
9,Furnished,2666,15.64%


## 🔎 أمثلة حية: عقارات تم اكتشاف مرافقها من الـ Description


### 🏠 Luxury Residence | Prime Location | Investor Deal
- **السعر:** `2,719,000`
- **✅ يحتوي على:** Swimming Pool
- **📝 مقتطف من الوصف:**
> *House & Hedges Real Estate is pleased to offer this Apartment in Hado by Beyond Developments at Siora on the Dubai Islands Hado by Beyond Developments at Siora on the Dubai Islands introduces an art of refined living within 1 to 4-bedroom apartments, simplexes, & duplexes. This masterpiece project d...*



### 🏠 Under Original Price | Exclusive | Sea View
- **السعر:** `5,500,000`
- **✅ يحتوي على:** Swimming Pool
- **📝 مقتطف من الوصف:**
> *This luxury beachfront 2-bedroom apartment is located in the prestigious The 8, Palm Jumeirah. Offering luxurious living with a prime location on the Palm Jumeirah, this fully furnished property is an ideal investment opportunity with excellent amenities and beach access. You can enjoy the hotel's c...*



### 🏠 STUIO FOR SELL DIRECT FROM DEVELOPER-NO COMMISSION
- **السعر:** `578,000`
- **✅ يحتوي على:** Swimming Pool
- **📝 مقتطف من الوصف:**
> *Studio Apartment for Sale | RR Grand, Dubai South Own a modern studio apartment in  RR Grand, Dubai South , a vibrant community offering excellent connectivity, contemporary design, and outstanding investment potential. Whether you're a first-time buyer or an investor, this property is an ideal choi...*


- check all amenities

In [20]:
import pandas as pd
import numpy as np
from collections import Counter
from IPython.display import display, Markdown

# ==========================================
# 🔍 استخراج كل المرافق الفريدة من عمود amenities
# ==========================================
display(Markdown("## 🔍 استخراج كل المرافق الفريدة من عمود `amenities`"))

if 'amenities' not in df.columns:
    display(Markdown("❌ عمود `amenities` غير موجود!"))
else:
    # 1. تقسيم النص إلى قائمة مرافق
    # البيانات مفصولة بـ ", " أو ","
    amenities_split = df['amenities'].astype(str).str.split(',')

    # 2. تسطيح القائمة (Flatten) وإزالة المسافات
    all_amenities = []
    for row in amenities_split:
        if isinstance(row, list):
            for amenity in row:
                cleaned = str(amenity).strip().title()
                if cleaned and cleaned != 'Nan' and cleaned != 'None' and len(cleaned) > 2:
                    all_amenities.append(cleaned)

    # 3. حساب التكرار
    amenity_counts = Counter(all_amenities)

    # 4. تحويل إلى DataFrame
    amenities_df = pd.DataFrame(
        amenity_counts.most_common(),
        columns=['Amenity', 'Count']
    )

    # إضافة نسبة مئوية
    total_properties = len(df)
    amenities_df['Percentage'] = (amenities_df['Count'] / total_properties * 100).round(2)

    # 5. عرض النتائج
    display(Markdown(f"###  إجمالي المرافق الفريدة: **{len(amenities_df)}**"))
    display(Markdown(f"**إجمالي العقارات:** {total_properties:,}"))

    # عرض كل المرافق
    display(amenities_df.style.format({
        'Count': '{:,}',
        'Percentage': '{:.2f}%'
    }).background_gradient(cmap='Blues', subset=['Count']))

    # 6. تصنيف المرافق حسب التكرار
    display(Markdown("###  تصنيف المرافق حسب التكرار"))

    high_freq = amenities_df[amenities_df['Percentage'] >= 30]
    medium_freq = amenities_df[(amenities_df['Percentage'] >= 10) & (amenities_df['Percentage'] < 30)]
    low_freq = amenities_df[(amenities_df['Percentage'] >= 1) & (amenities_df['Percentage'] < 10)]
    very_low = amenities_df[amenities_df['Percentage'] < 1]

    display(Markdown(f"""
    | الفئة | عدد المرافق | النسبة |
    |-------|-------------|--------|
    | 🔴 **عالية التكرار** (≥30%) | {len(high_freq)} | ستُستخدم كـ Features أساسية |
    | 🟡 **متوسطة التكرار** (10-30%) | {len(medium_freq)} | مفيدة للتحليل |
    | 🟢 **منخفضة التكرار** (1-10%) | {len(low_freq)} | قد تُدمج في فئة "أخرى" |
    |  **نادرة جداً** (<1%) | {len(very_low)} | قد تُحذف أو تُدمج |
    """))

    # 7. عرض المرافق عالية التكرار فقط
    if not high_freq.empty:
        display(Markdown("### 🔴 المرافق عالية التكرار (≥30%) - الأكثر أهمية"))
        display(high_freq.style.format({
            'Count': '{:,}',
            'Percentage': '{:.2f}%'
        }))

    # 8. عرض المرافق متوسطة التكرار
    if not medium_freq.empty:
        display(Markdown("### 🟡 المرافق متوسطة التكرار (10-30%)"))
        display(medium_freq.style.format({
            'Count': '{:,}',
            'Percentage': '{:.2f}%'
        }))

    # 9. عرض المرافق منخفضة التكرار (عينة)
    if not low_freq.empty:
        display(Markdown(f"### 🟢 المرافق منخفضة التكرار (1-10%) - أول 20 فقط"))
        display(low_freq.head(20).style.format({
            'Count': '{:,}',
            'Percentage': '{:.2f}%'
        }))

    # 10. عرض المرافق النادرة جداً (عينة)
    if not very_low.empty:
        display(Markdown(f"### ⚪ المرافق النادرة جداً (<1%) - أول 20 فقط"))
        display(very_low.head(20).style.format({
            'Count': '{:,}',
            'Percentage': '{:.2f}%'
        }))

    # 11. حفظ القائمة الكاملة في ملف
    amenities_df.to_csv('all_unique_amenities.csv', index=False)
    display(Markdown("✅ **تم حفظ القائمة الكاملة في ملف `all_unique_amenities.csv`**"))

## 🔍 استخراج كل المرافق الفريدة من عمود `amenities`

###  إجمالي المرافق الفريدة: **133**

**إجمالي العقارات:** 17,042

,Amenity,Count,Percentage
0,Balcony Or Terrace,"11,952",70.13%
1,Centrally Air-Conditioned,"10,955",64.28%
2,Kids Play Area,"10,283",60.34%
3,Swimming Pool,"9,650",56.62%
4,Security Staff,"9,458",55.50%
5,Gym Or Health Club,"9,319",54.68%
6,Barbeque Area,"8,444",49.55%
7,Lawn Or Garden,"8,369",49.11%
8,Lobby In Building,"8,278",48.57%
9,Cctv Security,"8,084",47.44%


###  تصنيف المرافق حسب التكرار

...

### 🟢 المرافق منخفضة التكرار (1-10%) - أول 20 فقط

,Amenity,Count,Percentage
46,Completion Year,"1,251",7.34%
47,Elevators In Building,"1,118",6.56%
48,: 2,"1,070",6.28%
49,Nearby Shopping Malls,"1,048",6.15%
50,Floor,639,3.75%
51,: 2026,574,3.37%
52,: 3,466,2.73%
53,: 4,433,2.54%
54,: 6,332,1.95%
55,: 5,276,1.62%


### ⚪ المرافق النادرة جداً (<1%) - أول 20 فقط

,Amenity,Count,Percentage
57,: 2028,156,0.92%
58,: 2027,142,0.83%
59,: 7,112,0.66%
60,: 2029,86,0.50%
61,: 16,76,0.45%
62,: 8,67,0.39%
63,Total Floors,46,0.27%
64,: 2025,40,0.23%
65,: 2024,29,0.17%
66,: 10,26,0.15%


✅ **تم حفظ القائمة الكاملة في ملف `all_unique_amenities.csv`**

### Zone_name


In [21]:
df.loc[df['zone_name'].isnull()]

,id,scraped_at,url,property_img,price,currency,title,location,beds,baths,area,property_type,purpose,reference_no,completion,furnishing,trucheck_date,added_on,handover_date,description,amenities,building_name,floors,retail_centres,swimming_pools,parking_spaces,building_area,elevators,agent_name,agency_name,developer,ownership,built_up_area,balcony_size,parking_availability,permit_number,zone_name,registered_agency,rera,brn,has_swimming_pool,has_gym,has_parking,has_security,has_concierge,has_kids_area,has_pets_allowed,has_maids_room,has_balcony,has_furnished,has_central_ac,has_sauna,has_steam_room,has_jacuzzi,has_garden,has_bbq_area,has_business_center,has_laundry,has_broadband,has_elevator,has_prayer_room,has_private_pool,has_shared_pool,amenities_count_from_desc
10,17033,2026-09-01 08:56:42.270964,https://www.bayut.com/property/details-1582195...,https://images.bayut.com/thumbnails/852230806-...,"1,700,000",AED,Corner villa for sale in Sharjah / Al Mansoura,"Al Mansoura, Sharjah",6,6,"6,000 sqft",Villa,For Sale,Bayut - 109017-9U9Rmo,Ready,Unfurnished,None,14 July 2026,None,Corner villa for sale in Sharjah (Al Mansoura)...,"Electricity Backup, Parking Spaces, : 5, Centr...",None,None,None,None,None,None,None,Sameh Mohammed,Burj Al Wadi Real Estate,None,None,None,None,None,None,None,None,None,None,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
22,17021,2026-09-01 08:54:42.695764,https://www.bayut.com/property/details-1621146...,https://images.bayut.com/thumbnails/860509539-...,"865,000",AED,Exceptional Opportunity to Own a Luxury Sea Vi...,"S1 Tower, Al Mamzar, Sharjah",1,2,"1,300 sqft",Apartment,For Sale,Bayut - 108275-1rHclc,Off-Plan,Unfurnished,None,18 August 2026,Q1 2029,Exceptional Opportunity to Own a Luxury Sea Vi...,"Centrally Air-Conditioned, Balcony or Terrace,...",None,None,None,None,None,None,None,Mohamad Kalid,Luxurious Seven Stars Real Estate,None,None,None,None,None,None,None,None,None,None,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2
28,17015,2026-09-01 08:53:37.313229,https://www.bayut.com/property/details-1571902...,https://images.bayut.com/thumbnails/849830558-...,"880,000",AED,Residential land for sale (G+2)prime location ...,"Al Zubair Orchards, Al Zubair, Sharjah",None,None,"6,110 sqft",Residential Plot,For Sale,Bayut - 9123-XtEVYD,Ready,None,None,5 July 2026,None,"Residential land for sale (G+2), prime locatio...",Freehold,None,None,None,None,None,None,None,Ali Mohamed Ali,Al Mutamaiz Real Estate,None,None,None,None,None,None,None,None,None,None,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
36,17007,2026-09-01 08:53:03.341426,https://www.bayut.com/property/details-1600303...,https://images.bayut.com/thumbnails/856123838-...,"650,001",AED,2 BHK FOR SALE IN Kentia Residence is A Modern...,"Kentia, Ajman Uptown, Ajman",2,2,"1,730 sqft",Apartment,For Sale,Bayut - 105326-EAfTec,Ready,Unfurnished,None,29 July 2026,None,Property Amenities & Features : The complex ba...,"Service Elevators, Gym or Health Club, Swimmin...",None,None,None,None,None,None,None,Dansh Mohammed,M. K Properties,None,None,None,None,None,None,None,None,None,None,1,1,1,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,6
40,17003,2026-09-01 08:52:44.762726,https://www.bayut.com/property/details-1570127...,https://images.bayut.com/thumbnails/855702224-...,"1,800,000",AED,Elegant and Stunning Villa For Sale In Al Helio,"Al Helio, Ajman",5,5,"3,121 sqft",Villa,For Sale,Bayut - HELIOVILLA-SHA,Ready,Unfurnished,None,3 July 2026,None,"Elegant and stunning villa in Al Helio, Ajman ...",None,None,None,None,None,None,None,None,Shameema Shammi,Best Homes Real Estate,None,None,None,None,None,None,None,None,None,None,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15979,1063,2026-08-30 18:14:50.951652,ht

- check if there are any info about zone name

In [22]:
df.loc[df['zone_name'].isnull()]['description'][10]

'Corner villa for sale in Sharjah (Al Mansoura). The villa features 6 bedrooms, 2 living areas, a dining room, and a separate external majlis. It also includes a spacious courtyard. Key features include a prime, upscale location; the villa faces two streets; it boasts exquisite classic finishes and external annexes; and it is priced below market value.'

In [23]:
df.loc[df['zone_name'].isnull()]['location'][10]

'Al Mansoura, Sharjah'

In [24]:
df['location'][2]

'RR Grand, Residential District, Dubai South, Dubai'

In [25]:
df.head(2)[['location','zone_name']]

,location,zone_name
0,"Hado by Beyond Tower C, Hado by Beyond, Dubai ...",Palm Deira
1,"Th8, The Crescent, Palm Jumeirah, Dubai",Palm Jumeirah


In [26]:
df.loc[df['zone_name'].isnull()]['url'][10]

'https://www.bayut.com/property/details-15821950.html'

**Note:**
- info not found, but can extract it from `location, description, title`

### furnishing

In [27]:
null_furnishing = df.loc[df['furnishing'].isnull()]
null_furnishing.head()

,id,scraped_at,url,property_img,price,currency,title,location,beds,baths,area,property_type,purpose,reference_no,completion,furnishing,trucheck_date,added_on,handover_date,description,amenities,building_name,floors,retail_centres,swimming_pools,parking_spaces,building_area,elevators,agent_name,agency_name,developer,ownership,built_up_area,balcony_size,parking_availability,permit_number,zone_name,registered_agency,rera,brn,has_swimming_pool,has_gym,has_parking,has_security,has_concierge,has_kids_area,has_pets_allowed,has_maids_room,has_balcony,has_furnished,has_central_ac,has_sauna,has_steam_room,has_jacuzzi,has_garden,has_bbq_area,has_business_center,has_laundry,has_broadband,has_elevator,has_prayer_room,has_private_pool,has_shared_pool,amenities_count_from_desc
27,17016,2026-09-01 08:53:49.794702,https://www.bayut.com/property/details-1571183...,https://images.bayut.com/thumbnails/849707176-...,"2,000,000",AED,LARGE 3BR+M | UPGRADED | PODIUM | DIRECT ACCES...,"Zahra Apartments 1A, Zahra Apartments, Town Sq...",3,4,"1,769 sqft",Apartment,For Sale,Bayut - Prime_SH_34,Ready,None,29 August 2026,4 July 2026,None,Prime Capital Real Estate is proud to present ...,"Balcony or Terrace, Lobby in Building, Service...",None,None,None,None,None,None,None,Shaad Haji,Prime Capital Real Estate,None,None,None,None,None,None,28153,None,None,29775,1,1,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,6
28,17015,2026-09-01 08:53:37.313229,https://www.bayut.com/property/details-1571902...,https://images.bayut.com/thumbnails/849830558-...,"880,000",AED,Residential land for sale (G+2)prime location ...,"Al Zubair Orchards, Al Zubair, Sharjah",None,None,"6,110 sqft",Residential Plot,For Sale,Bayut - 9123-XtEVYD,Ready,None,None,5 July 2026,None,"Residential land for sale (G+2), prime locatio...",Freehold,None,None,None,None,None,None,None,Ali Mohamed Ali,Al Mutamaiz Real Estate,None,None,None,None,None,None,None,None,None,None,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
34,17009,2026-09-01 08:53:09.276542,https://www.bayut.com/property/details-1584149...,https://images.bayut.com/thumbnails/852603235-...,"1,350,000",AED,SPECIOUS corner PLOT AL HELIO 1 AVAILBLE FOR SALE,"Al Helio 1, Al Helio, Ajman",None,None,"5,400 sqft",Residential Plot,For Sale,Bayut - 105099-4JawGf,Ready,None,None,15 July 2026,None,"WELCOME TO QSA PROPERTIES! Al Helio 2, Ajman. ...","Centrally Air-Conditioned, Central Heating, Do...",None,None,None,None,None,None,None,Sheraz Sultan,QSA Properties,None,None,None,None,None,None,124674,None,None,None,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
38,17005,2026-09-01 08:52:55.301580,https://www.bayut.com/property/details-1569481...,https://images.bayut.com/thumbnails/849391384-...,"1,600,000",AED,Prestige Collection | Villas Blending Accessib...,"Al Suyoh, Sharjah",3,4,"2,500 sqft",Villa,For Sale,Bayut - 101087-hkI2z6,Ready,None,None,3 July 2026,None,Elegant Family Living in Al Suyoh Step into el...,"Centrally Air-Conditioned, Gym or Health Club,...",None,None,None,None,None,None,None,Glenn Geo,Keyspace Real Estate,None,None,None,None,None,None,23427,None,None,None,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
48,16995,2026-09-01 08:51:54.534206,https://www.bayut.com/property/details-1442183...,https://images.bayut.com/thumbnails/824133060-...,"415,000",AED,"BIG TERRACE STUDIO I 1 % PAY ONLY AED 3,214 Mo...","Garden Residences, Emirates City, Ajman",Studio,1,920 sqft,Apartment,For Sale,Bayut - AZHA-GARDEN-27-02-26,Ready,None,None,27 February 2026,None,BIG TERRACE STUDIO Apartment for Sale in AZHA ...,"Parking Spaces, : 1, Centrally Air-Conditioned...",None,None,None,None,None,None,None,Muhammad Shahid,M. K Properties,None,None,None,None,None,None,None,None,None,None,1,1,1,0,0,1,0,0,1,1,0,0,0,0,1,0,0,0,0,0,1,0,0,8


- check if property type have a relation

In [28]:
null_furnishing['property_type'].unique()

array(['Apartment', 'Residential Plot', 'Villa', 'Townhouse', 'Penthouse',
       'Residential Building', 'Villa Compound'], dtype=object)

- check random url to see if there are any info.

In [29]:
null_furnishing['url'][90:95].values

array(['https://www.bayut.com/property/details-16350554.html',
       'https://www.bayut.com/property/details-16350778.html',
       'https://www.bayut.com/property/details-16350760.html',
       'https://www.bayut.com/property/details-16350761.html',
       'https://www.bayut.com/property/details-16350795.html'],
      dtype=object)

**Note:**
- after check some links, the info of furnising not found.


- check if there are info about furnishing

In [30]:
null_furnishing['description'][:1].values

array(['Prime Capital Real Estate is proud to present this beautiful terrace 3BR+M Apartment for rent in Zahra Apartments, in the Heart of Townsquare.  Property Features: -3 Bedrooms + M -4 Bathrooms -Bright and Spacious Layout -Master bedroom with en-suite bath and walk in wardrobe -semi closed Kitchen -upgraded wooden flooring -upgraded false ceiling with lighting.  -Powder Room -separate laundry room -Spacious Living Area -Large Terrace with direct access to the Pool and Courtyard -1 parking space -1769.26 sq ft -Vacant Features and Amenities: - Shared Swimming Pool - Fully equipped Shared Gym - \u2060Kids play area - Restaurants and Cafes - Supermarkets - Community Centre - Park - \u2060Medical clinics Zahra Apartments is one of the four apartment buildings of Zahra Apartments located in Town Square. It is a 7-storey residential building set in a quiet, perfectly landscaped community.'],
      dtype=object)

In [31]:
df.loc[df['furnishing']=='Furnished']['description'][:1].values

array(["This luxury beachfront 2-bedroom apartment is located in the prestigious The 8, Palm Jumeirah. Offering luxurious living with a prime location on the Palm Jumeirah, this fully furnished property is an ideal investment opportunity with excellent amenities and beach access. You can enjoy the hotel's comfort.  Building Amenities: •\tPrivate Beach Access •\tWater Sports •\tImmersive Infinity Pools •\tExclusive Family and Kids Pool •\tBasketball Court •\tDynamic Dining •\tAll-day Coffee Shop •\tKids Club •\tSpa and a state-of-the-art fitness center.  Features: •\tBasement parking •\tBalcony •\tView of Water •\tShared swimming pool •\tPrivate Garden •\tPets allowed •\tCentral air conditioning •\tGymnasium •\tKitchen Appliances •\tShops •\tSecurity The8 is one of Palm Jumeirah´s most luxurious projects, offering world-class amenities and services. Experience beachfront living at its finest in one of Dubai’s most prestigious addresses. Don’t miss this incredible opportunity to own a pi

In [32]:
import pandas as pd
import numpy as np
import re
from IPython.display import display, Markdown

# ==========================================
# 🔍 Furnished/Unfurnished Detective
# ==========================================
display(Markdown("## 🔍 Furnished/Unfurnished Detective"))
display(Markdown("""
البحث في **4 مصادر** عن أي دليل على حالة التأثيث:
1. عمود `furnishing` (المصدر الأساسي)
2. عمود `amenities` (أحياناً "Furnished" بييجي كمرفق)
3. عمود `description` (الوصف الطويل)
4. عمود `title` (العنوان)
"""))

# ==========================================
# 1️⃣ فحص عمود furnishing الأساسي
# ==========================================
display(Markdown("### 1️⃣ عمود `furnishing` (المصدر الأساسي)"))

if 'furnishing' in df.columns:
    furnishing_raw = df['furnishing'].astype(str).str.strip()

    # القيم الفاضية أو NaN
    empty_mask = furnishing_raw.isin(['', 'nan', 'None', 'NaN', 'N/A'])

    # القيم الفريدة
    unique_vals = furnishing_raw[~empty_mask].unique()

    display(Markdown(f"**القيم الفريدة الموجودة:** `{list(unique_vals)}`"))
    display(Markdown(f"**عدد القيم الفاضية:** `{empty_mask.sum():,}` من `{len(df):,}` "
                     f"({(empty_mask.sum()/len(df))*100:.1f}%)"))
else:
    display(Markdown("❌ عمود `furnishing` غير موجود!"))
    empty_mask = pd.Series([True] * len(df))

# ==========================================
# 2️⃣ البحث في عمود amenities
# ==========================================
display(Markdown("### 2️⃣ البحث في عمود `amenities`"))

furnished_patterns = [
    r'\bfurnished\b',
    r'\bfully furnished\b',
    r'\bsemi[- ]furnished\b',
    r'\bpartially furnished\b',
]

unfurnished_patterns = [
    r'\bunfurnished\b',
    r'\bnot furnished\b',
]

# البحث عن "Furnished" في amenities
amenities_text = df['amenities'].fillna('').astype(str).str.lower()

has_furnished_amenity = amenities_text.str.contains(
    '|'.join(furnished_patterns), regex=True, na=False
)
has_unfurnished_amenity = amenities_text.str.contains(
    '|'.join(unfurnished_patterns), regex=True, na=False
)

display(Markdown(f"- **عقارات فيها 'Furnished' في amenities:** `{has_furnished_amenity.sum():,}`"))
display(Markdown(f"- **عقارات فيها 'Unfurnished' في amenities:** `{has_unfurnished_amenity.sum():,}`"))

# أمثلة حية
furnished_amenity_samples = df[has_furnished_amenity]['amenities'].head(3).tolist()
if furnished_amenity_samples:
    display(Markdown("**📝 أمثلة:**"))
    for i, s in enumerate(furnished_amenity_samples):
        display(Markdown(f"  {i+1}. `{str(s)[:200]}`"))

# ==========================================
# 3️⃣ البحث في عمود description (الذهب!)
# ==========================================
display(Markdown("### 3️⃣ البحث في عمود `description` (الوصف الطويل)"))

desc_text = df['description'].fillna('').astype(str).str.lower()

has_furnished_desc = desc_text.str.contains(
    '|'.join(furnished_patterns), regex=True, na=False
)
has_unfurnished_desc = desc_text.str.contains(
    '|'.join(unfurnished_patterns), regex=True, na=False
)

display(Markdown(f"- **عقارات فيها 'Furnished' في description:** `{has_furnished_desc.sum():,}`"))
display(Markdown(f"- **عقارات فيها 'Unfurnished' في description:** `{has_unfurnished_desc.sum():,}`"))

# أمثلة حية من الوصف
furnished_desc_samples = df[has_furnished_desc & empty_mask]['description'].head(3).tolist()
if furnished_desc_samples:
    display(Markdown("**📝 أمثلة من عقارات فاضية في `furnishing` لكن الوصف بيقول 'Furnished':**"))
    for i, s in enumerate(furnished_desc_samples):
        # نبرز الكلمة المفتاحية
        highlighted = re.sub(
            r'(furnished|unfurnished)',
            r'**\1**',
            str(s)[:300],
            flags=re.IGNORECASE
        )
        display(Markdown(f"  {i+1}. {highlighted}..."))

# ==========================================
# 4️⃣ البحث في عمود title
# ==========================================
display(Markdown("### 4️⃣ البحث في عمود `title` (العنوان)"))

title_text = df['title'].fillna('').astype(str).str.lower()

has_furnished_title = title_text.str.contains(
    '|'.join(furnished_patterns), regex=True, na=False
)
has_unfurnished_title = title_text.str.contains(
    '|'.join(unfurnished_patterns), regex=True, na=False
)

display(Markdown(f"- **عقارات فيها 'Furnished' في title:** `{has_furnished_title.sum():,}`"))
display(Markdown(f"- **عقارات فيها 'Unfurnished' في title:** `{has_unfurnished_title.sum():,}`"))

# أمثلة حية
furnished_title_samples = df[has_furnished_title & empty_mask]['title'].head(5).tolist()
if furnished_title_samples:
    display(Markdown("**📝 أمثلة من العناوين:**"))
    for i, s in enumerate(furnished_title_samples):
        display(Markdown(f"  {i+1}. `{s}`"))

# ==========================================
# 📊 ملخص شامل: كم عقار ممكن نملأه؟
# ==========================================
display(Markdown("## 📊 ملخص شامل: إمكانيات الملء"))

# العقارات الفاضية في furnishing
empty_count = empty_mask.sum()

# من الفاضية، كام واحد فيه دليل "Furnished" في أي مصدر؟
can_fill_furnished = empty_mask & (
    has_furnished_amenity | has_furnished_desc | has_furnished_title
)
can_fill_unfurnished = empty_mask & (
    has_unfurnished_amenity | has_unfurnished_desc | has_unfurnished_title
)
no_clue = empty_mask & ~(
    has_furnished_amenity | has_furnished_desc | has_furnished_title |
    has_unfurnished_amenity | has_unfurnished_desc | has_unfurnished_title
)

summary = pd.DataFrame({
    'الحالة': [
        '✅ موجود بالفعل في furnishing',
        '🟢 يمكن ملؤه بـ Furnished (من مصادر أخرى)',
        '🔵 يمكن ملؤه بـ Unfurnished (من مصادر أخرى)',
        '⚪ لا يوجد أي دليل → افتراض Unfurnished',
    ],
    'العدد': [
        (~empty_mask).sum(),
        can_fill_furnished.sum(),
        can_fill_unfurnished.sum(),
        no_clue.sum(),
    ],
    'النسبة': [
        f"{((~empty_mask).sum()/len(df))*100:.1f}%",
        f"{(can_fill_furnished.sum()/len(df))*100:.1f}%",
        f"{(can_fill_unfurnished.sum()/len(df))*100:.1f}%",
        f"{(no_clue.sum()/len(df))*100:.1f}%",
    ]
})

display(summary.style.hide(axis='index').set_properties(**{
    'text-align': 'left',
    'font-family': 'Inter, sans-serif'
}))

# ==========================================
# 🎯 التوصية النهائية
# ==========================================
display(Markdown("## 🎯 التوصية النهائية للـ Pipeline"))
display(Markdown(f"""
### استراتيجية الملء (3 خطوات):

**الخطوة 1:** لو `furnishing` فيه قيمة → نخليها زي ما هي (بعد التنظيف)

**الخطوة 2:** لو `furnishing` فاضي → نبحث بالترتيب:
  1. `amenities` فيه "Furnished"؟ → **Furnished**
  2. `description` فيه "Furnished"؟ → **Furnished**
  3. `title` فيه "Furnished"؟ → **Furnished**
  4. أي مصدر فيه "Unfurnished"؟ → **Unfurnished**

**الخطوة 3:** لو مفيش أي دليل في أي مصدر → **Unfurnished** (افتراض تجاري)

> 💡 **النتيجة:** هنقدر نملأ **{can_fill_furnished.sum() + can_fill_unfurnished.sum() + no_clue.sum():,}**
> قيمة فاضية من أصل **{empty_count:,}** ({((can_fill_furnished.sum() + can_fill_unfurnished.sum() + no_clue.sum())/max(empty_count,1))*100:.0f}% تغطية)
"""))

## 🔍 Furnished/Unfurnished Detective


البحث في **4 مصادر** عن أي دليل على حالة التأثيث:
1. عمود `furnishing` (المصدر الأساسي)
2. عمود `amenities` (أحياناً "Furnished" بييجي كمرفق)
3. عمود `description` (الوصف الطويل)
4. عمود `title` (العنوان)


### 1️⃣ عمود `furnishing` (المصدر الأساسي)

**القيم الفريدة الموجودة:** `['Unfurnished', 'Furnished']`

**عدد القيم الفاضية:** `3,286` من `17,042` (19.3%)

...

  5. `Brand new villa, first occupancy, fully furnished and ready to move in immediately. Price includes registration and ownership fees. Freehold ownership`

## 📊 ملخص شامل: إمكانيات الملء

الحالة,العدد,النسبة
✅ موجود بالفعل في furnishing,13756,80.7%
🟢 يمكن ملؤه بـ Furnished (من مصادر أخرى),124,0.7%
🔵 يمكن ملؤه بـ Unfurnished (من مصادر أخرى),26,0.2%
⚪ لا يوجد أي دليل → افتراض Unfurnished,3141,18.4%


## 🎯 التوصية النهائية للـ Pipeline


### استراتيجية الملء (3 خطوات):

**الخطوة 1:** لو `furnishing` فيه قيمة → نخليها زي ما هي (بعد التنظيف)

**الخطوة 2:** لو `furnishing` فاضي → نبحث بالترتيب:
  1. `amenities` فيه "Furnished"؟ → **Furnished**
  2. `description` فيه "Furnished"؟ → **Furnished**
  3. `title` فيه "Furnished"؟ → **Furnished**
  4. أي مصدر فيه "Unfurnished"؟ → **Unfurnished**

**الخطوة 3:** لو مفيش أي دليل في أي مصدر → **Unfurnished** (افتراض تجاري)

> 💡 **النتيجة:** هنقدر نملأ **3,291**
> قيمة فاضية من أصل **3,286** (100% تغطية)
